# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an example workflow for loading and exploring the FAIR$^{2}$ colorectal cancer survivors dataset using the `mlcroissant` library. All schema entities (record sets, fields, or columns) are referenced by their `@id`.

### Dataset Source
The dataset is described and distributed via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load Croissant metadata and tabular records from the dataset using `mlcroissant`. This retrieves interoperable metadata and records described in the schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Dataset object from the Croissant schema URL
dataset = mlc.Dataset(croissant_url)

# Retrieve the dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nLicense: {metadata.license}")

## 2. Data Overview

Explore the available record sets and their fields by `@id`. This provides a high-level schema of dataset tables and variables.

Let's enumerate all available record sets (`cr:RecordSet`) and, for each, list their corresponding field and column `@id`s.

In [ ]:
# List available record sets with field and column IDs
record_sets = list(dataset.record_sets)

print("Available record sets, fields, and columns (@id):\n")
for rs in record_sets:
    print(f"Record set: {rs['@id']} ({rs.get('name', '')})")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for f in fields:
            if isinstance(f, dict):
                field_id = f.get('@id')
                field_name = f.get('name', '')
                print(f"  Field:   {field_id} ({field_name})")
                # Under each field, list the columns if specified
                if 'column' in f:
                    cols = f['column'] if isinstance(f['column'], list) else [f['column']]
                    for c in cols:
                        if isinstance(c, dict):
                            print(f"    Column: {c.get('@id')} ({c.get('name', '')})")
                        else:
                            print(f"    Column: {c}")
            else:
                print(f"  Field:   {f}")
    else:
        print("  No fields listed.")
    print()

## 3. Data Extraction

Now, load the records of each record set into a Pandas DataFrame. Use the `@id` of each record set.

If you saw one or more record sets in the overview above, set their `@id`s in the list below to load all their rows for analysis.

In [ ]:
# List all record set `@id`s (update according to the printed @ids above)
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
print('Loading records for each record set:')
for record_set_id in record_set_ids:
    print(f"  - {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records):
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"    Loaded {len(df)} rows; columns (field/column @ids): {list(df.columns)}")
    else:
        print("    No records found.")
        dataframes[record_set_id] = pd.DataFrame()

# Pick a record set for demonstration (first with data)
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id:
    print(f"\nColumns in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes with records were loaded.")

## 4. Exploratory Data Analysis (EDA)

Perform typical EDA tasks by referencing fields/columns via their `@id`. We'll select a numeric `@id` (field or column) and a grouping/categorical field (again referenced by their `@id`), filter rows, normalize, and group values.

In [ ]:
# --- Customize: set these to @id values printed earlier for the main table ---
# EXAMPLE placeholders that you should update as appropriate using IDs from above. For demonstration, use the first numeric column if found.
df = dataframes[main_record_set_id]
numeric_field_id = None
group_field_id = None

for col in df.columns:
    # Try to infer numeric columns
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

for col in df.columns:
    # Try to pick a grouping/categorical variable different from the numeric
    if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
        group_field_id = col
        break

print(f"Numeric field chosen for EDA: {numeric_field_id}")
print(f"Grouping field chosen: {group_field_id}")

# EDA: Filter, normalize, group
if numeric_field_id:
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = numeric_series.mean() if not np.isnan(numeric_series.mean()) else 0
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} rows")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series[numeric_series > threshold] - numeric_series.mean()) / numeric_series.std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by a categorical field
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No numeric field @id found for EDA.')

## 5. Visualization

Visualize value distributions or relationships between fields using `matplotlib`. All axes are labeled using `@id` values, as per the Croissant schema.

In [ ]:
if numeric_field_id:
    plt.figure(figsize=(8, 5))
    # Histogram of the numeric field
    plt.hist(df[numeric_field_id].dropna().astype(float), bins=15, color='skyblue', edgecolor='black')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.show()
    
    # If grouping field exists, boxplot/group plot
    if group_field_id:
        plt.figure(figsize=(10, 6))
        df_box = df[[group_field_id, numeric_field_id]].dropna()
        # Only plot if reasonably few groups
        n_groups = df_box[group_field_id].nunique()
        if n_groups > 1 and n_groups <= 12:
            df_box["_g"] = df_box[group_field_id].astype(str)
            df_box[numeric_field_id] = pd.to_numeric(df_box[numeric_field_id], errors='coerce')
            df_box.boxplot(column=numeric_field_id, by="_g", rot=45)
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.suptitle("")
            plt.tight_layout()
            plt.show()

## 6. Conclusion

Using the `mlcroissant` library and the Croissant schema, we loaded, explored, and visualized clinicopathological records of second primary colorectal cancer survivors. The dataset's schema-driven metadata enables robust referencing of variables for filtering, transformation, and grouped summarization—all using persistent `@id`s. This approach facilitates reproducibility and interoperability for downstream analysis.

**Next steps** might include advanced clinical modeling, further statistical tests, or integrating rich metadata annotations directly from the Croissant schema for downstream ML workflows.